# Tinjauan transform

Satu bagian per langkah transform. Di setiap bagian:
1. **Kode**: isi fungsi dari berkas `.py` (bukan salinan, jadi selalu yang terbaru).
2. **Hasil**: fungsi itu dijalankan pada data asli di `data_lake`.

Jalankan sel **Persiapan** dulu, lalu bagian yang ingin ditinjau.

In [1]:
# Persiapan
import glob
import importlib
import inspect
import json
import os
import sys
from pathlib import Path

# notebook ini ada di data-science/transform/; kerja dari data-science/
# supaya `from transform import ...` dan path data_lake/ jalan
akar = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "transform" / "parser.py").exists())
os.chdir(akar)
if str(akar) not in sys.path:
    sys.path.insert(0, str(akar))

import polars as pl
from IPython.display import HTML, display
from pygments import highlight
from pygments.formatters import HtmlFormatter
from pygments.lexers import PythonLexer

from transform import parser

importlib.reload(parser)  # ambil versi .py terbaru tanpa restart kernel
pl.Config.set_tbl_rows(-1)
pl.Config.set_fmt_str_lengths(90)


def tampilkan_kode(objek):
    """Kode sumber modul/fungsi dengan warna tema gelap (github-dark)."""
    gaya = HtmlFormatter(style="github-dark", noclasses=True, nobackground=True)
    sumber = objek if isinstance(objek, str) else inspect.getsource(objek)
    isi = highlight(sumber, PythonLexer(), gaya)
    display(HTML(f'<div style="background:#0d1117;padding:12px;border-radius:6px">{isi}</div>'))

## Task 1A: berat (`transform/parser.py`)

Aturan: spec bagian 5.2. Fungsi yang dipakai transform nanti:
- `berat_gram(teks)`: berat dalam gram dari nama / variant / slug, atau `None`.
- `satuan_baris(nama, variant, banyak_opsi, usp)`: berat + sumber + jenis satuan + jumlah untuk satu baris. Produk **beberapa opsi** hanya membaca `variant` (A1).
- `pilih_berat(kandidat)`: sumber pertama yang punya berat (Blibli: `variant` → `nama`).
- `berat_dari_slug(url)`: cadangan Tokopedia kalau nama tanpa berat.
- `mangga_manalagi(nama)`: memuat mangga (atau ejaan lain: manga, mango, m4ngg4, …) **dan** manalagi.
- `jumlah_buah(nama)`, `jenis_satuan(nama, berat)`: `berat` / `jumlah` / `kemasan` / `lain`.

### Kode

In [2]:
tampilkan_kode(parser)

### Hasil: semua nama Mangga Manalagi unik (Blibli + Tokopedia, semua tanggal)

`cadangan` = `variant` (Blibli) atau URL tanpa query (Tokopedia).

In [3]:
baris = []
for berkas in glob.glob("data_lake/raw/blibli/date=*/*_page_*.json"):
    data = json.load(open(berkas, encoding="utf-8")).get("data") or {}
    for p in data.get("products") or []:
        if parser.mangga_manalagi(p.get("name")):
            baris.append({"sumber": "blibli", "nama": p["name"], "cadangan": p.get("variant") or "",
                           "banyak_opsi": bool(p.get("hasVariants")), "usp": p.get("uniqueSellingPoint")})
for berkas in glob.glob("data_lake*/raw/tokopedia/date=*/page_*.json"):
    try:
        produk = json.load(open(berkas, encoding="utf-8"))[0]["data"]["searchProductV5"]["data"]["products"]
    except (ValueError, KeyError, IndexError, TypeError):
        continue
    for p in produk:
        if parser.mangga_manalagi(p.get("name")):
            baris.append({"sumber": "tokopedia", "nama": p["name"], "cadangan": (p.get("url") or "").split("?")[0],
                           "banyak_opsi": False, "usp": None})


def hitung(b):
    if b["sumber"] == "blibli":
        s = parser.satuan_baris(b["nama"], b["cadangan"], b["banyak_opsi"], b["usp"])
        berat, sumber_berat, jenis = s.berat_gram, s.sumber_berat, s.jenis_satuan
    else:
        berat, sumber_berat = parser.pilih_berat([("nama", b["nama"])])
        if berat is None:
            berat = parser.berat_dari_slug(b["cadangan"])
            sumber_berat = "url" if berat is not None else None
        jenis = parser.jenis_satuan(b["nama"], berat)
    return {**{k: v for k, v in b.items() if k != "usp"},
            "berat_gram": berat, "sumber_berat": sumber_berat, "jenis_satuan": jenis}


berat = pl.DataFrame([hitung(b) for b in baris]).unique().sort("sumber", "jenis_satuan", "nama")
display(berat.group_by("sumber", "jenis_satuan").len().sort("sumber", "jenis_satuan"))
berat

sumber,jenis_satuan,len
str,str,u32
"""blibli""","""berat""",8
"""tokopedia""","""berat""",34
"""tokopedia""","""lain""",7


sumber,nama,cadangan,banyak_opsi,berat_gram,sumber_berat,jenis_satuan
str,str,str,bool,i64,str,str
"""blibli""","""Buah Mangga Manalagi segar""","""1/2 kg""",true,500,"""variant""","""berat"""
"""blibli""","""MANGGA MANALAGI 170 - 180 G""","""""",false,175,"""nama""","""berat"""
"""blibli""","""MANGGA MANALAGI 550-570 GR""","""""",false,560,"""nama""","""berat"""
"""blibli""","""Mangga Manalagi""","""1 kg""",false,1000,"""variant""","""berat"""
"""blibli""","""Mangga Manalagi 1 kg ( 0.9 - 1.1 kg / pack)""","""""",false,1000,"""nama""","""berat"""
"""blibli""","""Mangga Manalagi 500 gram ( 400 - 500 gram / pack)""","""""",false,500,"""nama""","""berat"""
"""blibli""","""Mangga Manalagi Buah Segar Ibaza fruitsBOX - 1 kg Buah Mangga Manalagi""","""""",false,1000,"""nama""","""berat"""
"""blibli""","""Mangga manalagi segar manis 1/2 kg""","""""",false,500,"""nama""","""berat"""
"""tokopedia""","""1 Kg Mangga MANALAGI MADU Manis Legit dikirim Mengkal""","""https://www.tokopedia.com/murahmurahkediri3/1-kg-mangga-manalagi-madu-manis-legit-dikirim-…",false,1000,"""nama""","""berat"""


### Hasil per opsi: produk mangga Blibli yang punya varian (tanggal terbaru)

Satu nama, beberapa opsi (`item_sku`), harga berbeda. **Keputusan A1 (8 Nov):** berat/jumlah
hanya dari `variant` opsi itu, **tidak pernah dipinjam dari nama** ("Buy 1 Get 1" → `lain`). `harga_per_kg` = pratinjau Task 7.
Data opsi baru ada sejak 7–8 Nov.

In [4]:
opsi = []
for berkas in sorted(glob.glob("data_lake/raw/blibli/date=*/varian/ps--*.json")):
    ringkasan = json.load(open(berkas, encoding="utf-8"))["data"]
    if "mangga" not in (ringkasan.get("name") or "").lower():
        continue
    for o in ringkasan.get("options") or []:
        kode = o["id"].split("_")[0]                      # "is--SKU_PP-123" -> "is--SKU"
        if o.get("selected"):
            harga = ringkasan["price"]["offered"]         # opsi default: harga ada di ps--
        else:
            berkas_opsi = Path(berkas).parent / f"{kode}.json"
            harga = (json.load(open(berkas_opsi, encoding="utf-8"))["data"]["price"]["offered"]
                     if berkas_opsi.exists() else None)
        variant = o["attributes"][0]["value"] if o.get("attributes") else None
        s = parser.satuan_baris(ringkasan["name"], variant, banyak_opsi=True)
        opsi.append({
            "tanggal": berkas.split("date=")[1][:10],
            "nama": ringkasan["name"],
            "item_sku": kode.removeprefix("is--"),
            "opsi_default": bool(o.get("selected")),
            "variant": variant,
            "harga": harga,
            "berat_gram": s.berat_gram,
            "sumber_berat": s.sumber_berat,
            "jenis_satuan": s.jenis_satuan,
            "jumlah_buah": s.jumlah_buah,
            "harga_per_kg": round(harga / s.berat_gram * 1000) if harga and s.berat_gram else None,
            "harga_per_buah": round(harga / s.jumlah_buah) if harga and s.jumlah_buah else None,
        })

opsi = pl.DataFrame(opsi)
opsi.filter(pl.col("tanggal") == pl.col("tanggal").max()).sort("nama", "berat_gram")

tanggal,nama,item_sku,opsi_default,variant,harga,berat_gram,sumber_berat,jenis_satuan,jumlah_buah,harga_per_kg,harga_per_buah
str,str,str,bool,str,f64,i64,str,str,null,i64,null
"""2026-11-09""","""Buah Mangga Indramayu""","""DOD-70072-00729-00002""",true,"""1/2 kg""",13500.0,500,"""variant""","""berat""",null,27000,null
"""2026-11-09""","""Buah Mangga Indramayu""","""DOD-70072-00729-00001""",false,"""1 kg""",20000.0,1000,"""variant""","""berat""",null,20000,null
"""2026-11-09""","""Buah Mangga Kiojay (Qyojay)""","""DOD-70072-00898-00002""",true,"""500g""",22000.0,500,"""variant""","""berat""",null,44000,null
"""2026-11-09""","""Buah Mangga Kiojay (Qyojay)""","""DOD-70072-00898-00001""",false,"""1kg""",40000.0,1000,"""variant""","""berat""",null,40000,null
"""2026-11-09""","""Buah Mangga Kiojay (Qyojay) Segar""","""DOD-70072-00667-00002""",true,"""500g""",22000.0,500,"""variant""","""berat""",null,44000,null
"""2026-11-09""","""Buah Mangga Kiojay (Qyojay) Segar""","""DOD-70072-00667-00001""",false,"""1kg""",40000.0,1000,"""variant""","""berat""",null,40000,null
"""2026-11-09""","""Buah Mangga Kweni Segar""","""DOD-70072-00692-00002""",true,"""500g""",13000.0,500,"""variant""","""berat""",null,26000,null
"""2026-11-09""","""Buah Mangga Kweni Segar""","""DOD-70072-00692-00001""",false,"""1000g""",25000.0,1000,"""variant""","""berat""",null,25000,null
"""2026-11-09""","""Buah Mangga Manalagi segar""","""DOD-70072-00076-00001""",true,"""1/2 kg""",13000.0,500,"""variant""","""berat""",null,26000,null


### Yang perlu ditinjau (3 pertanyaan, ±5 menit)

1. **Tanpa berat:** benarkah tidak ada berat tertulis di nama-nama ini?
2. **Nama sulit** (rentang, pecahan, garis miring): benarkah `berat_gram`-nya?
3. **Opsi yang mencurigakan:** opsi yang `harga_per_kg`-nya beda > 30% dari opsi lain di produk yang sama
   (tanda berat salah). Tabel kosong = tidak ada yang mencurigakan.

In [5]:
display(berat.filter(pl.col("jenis_satuan") != "berat").select("sumber", "nama"))
display(berat.filter(pl.col("nama").str.contains(r"\d\s*-\s*\d|/")).select("sumber", "nama", "berat_gram"))
terbaru = opsi.filter(pl.col("tanggal") == pl.col("tanggal").max())
terbaru.with_columns(median_produk=pl.col("harga_per_kg").median().over("nama")).filter(
    (pl.col("harga_per_kg") / pl.col("median_produk") - 1).abs() > 0.3
).select("nama", "variant", "harga", "berat_gram", "harga_per_kg", "median_produk")

sumber,nama
str,str
"""tokopedia""","""MANGGA MANALAGI MADU MANGGA MANALAGI MATANG MANIS Buah"""
"""tokopedia""","""Mangga Madu Manalagi Fresh - Buah Segar dengan Rasa Manis dan Nikmat"""
"""tokopedia""","""Mangga Manalagi Potong Manis"""
"""tokopedia""","""Mangga Manalagi Potong Manis"""
"""tokopedia""","""mangga manalagi jumbo isi 2 sampai 3 mentah tua rasa manis mantap"""
"""tokopedia""","""mangga manalagi jumbo isi 2 sampai 3 mentah tua rasa manis mantap"""
"""tokopedia""","""mangga manalagi mentah tua rasa manis dam mantap"""


sumber,nama,berat_gram
str,str,i64
"""blibli""","""MANGGA MANALAGI 170 - 180 G""",175
"""blibli""","""MANGGA MANALAGI 550-570 GR""",560
"""blibli""","""Mangga Manalagi 1 kg ( 0.9 - 1.1 kg / pack)""",1000
"""blibli""","""Mangga Manalagi 500 gram ( 400 - 500 gram / pack)""",500
"""blibli""","""Mangga manalagi segar manis 1/2 kg""",500
"""tokopedia""","""Mangga Madu keraton/Manalagi muda 1kg""",1000
"""tokopedia""","""Mangga Manalagi /1 kg""",1000
"""tokopedia""","""Mangga Manalagi Madu (Arum ) Tua Pohon asli Situbondo, isi 3-5/ kg - Buah""",1000
"""tokopedia""","""Mangga manalagi madu (Arum )fres & segar tua pohon 1kg isi 3-5 buah.""",1000


nama,variant,harga,berat_gram,harga_per_kg,median_produk
str,str,f64,i64,i64,f64


### Coba nama sendiri

Ganti teks di bawah lalu jalankan. Produk **tanpa opsi**: kosongkan `contoh_variant` (`""`).

In [6]:
contoh_nama = "Mangga Manalagi 900g - 1 Kg"
contoh_variant = "1 kg"   # "" kalau produk tanpa opsi

berat_contoh, sumber_contoh = parser.pilih_berat([("variant", contoh_variant), ("nama", contoh_nama)])
berat_contoh, sumber_contoh, parser.jenis_satuan(contoh_nama, berat_contoh)

(1000, 'variant', 'berat')

## Task 1B: jenis buah, olahan, asal, terjual, region (`transform/parser.py`)

- `jenis_buah(nama)`: kata kunci → `mangga`, `apel`, …, `lainnya`. Mangga ikut aturan salah ketik.
- `olahan(nama)`: penanda produk olahan (tanpa `sale` dan `pohon`, keputusan 8 Nov).
- `asal_dari_nama(nama)`: `impor` / `lokal` / `tidak_diketahui` (untuk Tokopedia).
- `terjual_dari_label(label)`, `ember_terjual(n)`: angka terjual Tokopedia / Blibli disetarakan.
- `region(kota)`: `jabodetabek` / `jawa_lain` / `luar_jawa` / `tidak_diketahui`.
- Multipack (`berat_dikali`): "8x125GR" → 1000 g, dicatat di karantina nanti (cek H).

### Kode

In [7]:
sumber = inspect.getsource(parser)
tampilkan_kode(sumber[sumber.index("# ---- Task 1B ----"):])

### Hasil (tanggal terbaru, Blibli + Tokopedia)

In [8]:
tanggal_baru = max(Path(f).name for f in glob.glob("data_lake/raw/blibli/date=*"))
produk = []
for berkas in glob.glob(f"data_lake/raw/blibli/{tanggal_baru}/*_page_*.json"):
    label_asal = "lokal" if "BU-1000190" in berkas else "impor"
    for p in (json.load(open(berkas, encoding="utf-8")).get("data") or {}).get("products") or []:
        produk.append({"sumber": "blibli", "nama": p["name"], "kota": p.get("location"),
                       "label_asal": label_asal, "label_terjual": None})
for berkas in glob.glob(f"data_lake/raw/tokopedia/{tanggal_baru}/page_*.json"):
    try:
        isi = json.load(open(berkas, encoding="utf-8"))[0]["data"]["searchProductV5"]["data"]["products"]
    except (ValueError, KeyError, IndexError, TypeError):
        continue
    for p in isi:
        label = next((l.get("title") for l in p.get("labelGroups") or []
                      if l.get("position") == "ri_product_credibility"), None)
        produk.append({"sumber": "tokopedia", "nama": p["name"], "kota": p["shop"].get("city"),
                       "label_asal": None, "label_terjual": label})

lain = pl.DataFrame(produk, infer_schema_length=None).with_columns(
    jenis_buah=pl.col("nama").map_elements(parser.jenis_buah, return_dtype=pl.Utf8),
    olahan=pl.col("nama").map_elements(parser.olahan, return_dtype=pl.Boolean),
    asal=pl.col("nama").map_elements(parser.asal_dari_nama, return_dtype=pl.Utf8),
    region=pl.col("kota").map_elements(parser.region, return_dtype=pl.Utf8, skip_nulls=False),
    terjual=pl.col("label_terjual").map_elements(parser.terjual_dari_label, return_dtype=pl.Int64),
    berat_gram=pl.col("nama").map_elements(parser.berat_gram, return_dtype=pl.Int64),
    berat_dikali=pl.col("nama").map_elements(parser.berat_dikali, return_dtype=pl.Boolean),
)
print(tanggal_baru, lain.height, "baris")
display(lain.group_by("jenis_buah").len().sort("len", descending=True))
display(lain.group_by("region").len().sort("len", descending=True))

date=2026-11-09 7600 baris


jenis_buah,len
str,u32
"""lainnya""",2800
"""jeruk""",1164
"""apel""",531
"""anggur""",423
"""mangga""",406
"""pir""",387
"""pisang""",359
"""jambu""",203
"""nanas""",195


region,len
str,u32
"""jabodetabek""",5499
"""jawa_lain""",1752
"""luar_jawa""",348
"""tidak_diketahui""",1


### Yang perlu ditinjau

1. **Olahan:** apakah baris-baris ini memang bukan buah utuh?
2. **Asal (Blibli punya label asli):** baris yang aturannya salah.
3. **Multipack:** apakah hasil perkaliannya benar?
4. **Kota luar Jawa:** adakah kota Jawa yang salah masuk ke sini?

In [9]:
display(lain.filter(pl.col("olahan")).select("sumber", "nama").unique().sort("nama").head(40))
display(lain.filter(pl.col("label_asal").is_not_null() & (pl.col("asal") != "tidak_diketahui")
                    & (pl.col("asal") != pl.col("label_asal"))).select("nama", "label_asal", "asal").unique())
display(lain.filter(pl.col("berat_dikali")).select("sumber", "nama", "berat_gram").unique())
lain.filter(pl.col("region") == "luar_jawa").select("kota").unique().sort("kota")

sumber,nama
str,str
"""tokopedia""","""(1000 GRAM) MANISAN BASAH ANEKA BUAH ASLI CIANJUR ANGGUR BOGOR SALAK MANGGA KEDONDONG BANG…"
"""tokopedia""","""-DRIED FRUIT ORANGE SLICE- BUAH SUNKIS SLICE KERING"""
"""tokopedia""","""1 KG MANISAN BUAH KHAS CIANJUR MANGGA SALAK PEPAYA CANAR KEDONGDONG LOKAL KEDONGDONG BANGK…"
"""tokopedia""","""1 KG MANISAN MANGGA BASAH / ASINAN BUAH MANGA AIR SEGAR / FRESH MANGO"""
"""tokopedia""","""1 Kg Buah Pinang Tua Segar Pilihan Terbaik Isi Sekitar 15-30 Biji | Buah Pinang Muda Fress…"
"""tokopedia""","""100 gr Dried Papaya / Buah Pepaya Kering / Dry Pepaya 100gr 250gr"""
"""tokopedia""","""100gr Gojiberi Kering - Dried Gojiberry - Kichi"""
"""tokopedia""","""1KG MANISAN BUAH LICO MERAH / LIKO KUNING CRUNCHY"""
"""tokopedia""","""1KG Manga Basah / Manisan Buah Mangga Segar / Fresh Wet Mango"""


nama,label_asal,asal
str,str,str
"""APEL ENVY SUPER 540 - 600 GR""","""lokal""","""impor"""
"""PEAR GOLDEN 730-750 GR""","""lokal""","""impor"""
"""Kepok Kuning""","""impor""","""lokal"""
"""Rumah Sayur Express Sunkist [500 gram]""","""lokal""","""impor"""
"""My Fruit Surabaya Jeruk Santang Lokal Buah [800gr]""","""impor""","""lokal"""
"""Buah Kiwi 1kg / Buah Kiwi Segar / Buah Kiwi Murah - Sahabatdapur""","""lokal""","""impor"""
"""Lemon Impor 500 gram (450 - 550 gram / pack)""","""lokal""","""impor"""
"""PEAR PACKHAM AFRICA PACK""","""lokal""","""impor"""
"""Anggur Hijau Shine Muscat Seedless (475 - 500 gram / pack)""","""lokal""","""impor"""


sumber,nama,berat_gram
str,str,i64
"""tokopedia""","""Jeruk Shantang Yuji Royal 500 Grm X 10 Pack""",5000
"""tokopedia""","""Jeruk Shantang Daun Jaring Manis Isi 8 X 700 Grm / Box""",5600
"""blibli""","""Sunsweet Buah Prunes 200 gr×12 Pcs""",2400
"""tokopedia""","""kurma Medjoel Palestina Al rward Large kemasan 1 kg x 8 pac ( 8 kg )""",8000
"""blibli""","""BLUEBERRY TRAIN TOY 8x125GR PACK""",1000
"""tokopedia""","""Kurma Akram Bam Mazafati 550g x 12 (1 dus)""",6600


kota
str
"""Balikpapan"""
"""Banda Aceh"""
"""Bandar Lampung"""
"""Banjarbaru"""
"""Banjarmasin"""
"""Batam"""
"""Bengkulu"""
"""Binjai"""
"""Bukittinggi"""


## Task 4: transform Blibli (`transform/blibli.py`)

Semua berkas mentah Blibli satu tanggal → tabel `blibli_produk`, **satu baris per opsi** (`item_sku`).
- baris default dari kartu hasil cari, baris opsi lain dari `varian/is--*.json`;
- kolom produk (penjual, kategori, rating) disalin ke baris opsi; kolom khusus kartu (voucher, kampanye) NULL;
- opsi tanpa harga (tidak tersedia) tetap ada dengan `harga` NULL (keputusan B).

### Kode

In [10]:
from transform import blibli
importlib.reload(blibli)
tampilkan_kode(blibli)

### Hasil: semua tanggal

In [11]:
from datetime import date

blibli_produk = pl.concat([
    blibli.ubah(folder, date.fromisoformat(folder.name.removeprefix("date=")))
    for folder in sorted(Path("data_lake/raw/blibli").glob("date=*"))
])
print(blibli_produk.height, "baris,", blibli_produk.width, "kolom")
blibli_produk.group_by("tanggal").agg(
    baris=pl.len(),
    opsi_lain=(~pl.col("opsi_default")).sum(),
    harga_null=pl.col("harga").is_null().sum(),
    berat_terbaca=pl.col("berat_gram").is_not_null().sum(),
).sort("tanggal")

26535 baris, 34 kolom


tanggal,baris,opsi_lain,harga_null,berat_terbaca
date,u32,u32,u32,u32
2026-10-18,1600,0,0,1060
2026-10-19,1600,0,0,1076
2026-10-27,1600,0,0,1066
2026-10-28,1600,0,0,1065
2026-10-29,1600,0,0,1068
2026-10-30,1599,0,0,1070
2026-10-31,1600,0,0,1068
2026-11-01,1599,0,0,1063
2026-11-02,1600,0,0,1071


### Yang perlu ditinjau

1. **Mangga Manalagi, tanggal terbaru:** satu baris per opsi, harga dan berat benar?
2. **Opsi tanpa harga:** benarkah ini opsi yang tidak tersedia?
3. **Satu produk lengkap** (semua 34 kolom) untuk dicek dengan halaman Blibli.

In [12]:
terbaru = blibli_produk.filter(pl.col("tanggal") == pl.col("tanggal").max())
display(terbaru.filter(pl.col("nama").map_elements(parser.mangga_manalagi, return_dtype=pl.Boolean))
        .select("item_sku", "opsi_default", "nama", "variant", "harga", "harga_coret", "berat_gram",
                "sumber_berat", "penjual", "url"))
display(terbaru.filter(pl.col("harga").is_null()).select("item_sku", "nama", "variant", "penjual").head(15))
terbaru.filter(pl.col("produk_id") == "DOD-70072-00076").transpose(include_header=True)

item_sku,opsi_default,nama,variant,harga,harga_coret,berat_gram,sumber_berat,penjual,url
str,bool,str,str,i64,i64,i64,str,str,str
"""RAM-70107-29314-00001""",true,"""MANGGA MANALAGI 170 - 180 G""",null,6800,6800,175,"""nama""","""Ranch Market Flagship Store""","""https://www.blibli.com/p/mangga-manalagi-170-180-g/ps--RAM-70107-29314?ds=RAM-70107-29314-…"
"""DOD-70072-00076-00001""",true,"""Buah Mangga Manalagi segar""","""1/2 kg""",13000,13000,500,"""variant""","""Doyan Daging Store Depok""","""https://www.blibli.com/p/buah-mangga-manalagi-segar/ps--DOD-70072-00076?ds=DOD-70072-00076…"
"""SAX-70009-02879-00001""",true,"""Mangga Manalagi""","""1 kg""",25700,25700,1000,"""variant""","""Sayurbox Flagship Store""","""https://www.blibli.com/p/mangga-manalagi/ps--SAX-70009-02879?ds=SAX-70009-02879-00001"""
"""DOD-70072-00076-00002""",false,"""Buah Mangga Manalagi segar""","""1 kg""",25000,25000,1000,"""variant""","""Doyan Daging Store Depok""","""https://www.blibli.com/p/buah-mangga-manalagi-segar/ps--DOD-70072-00076?ds=DOD-70072-00076…"


item_sku,nama,variant,penjual
str,str,str,str
"""BAF-70081-00137-00003""","""Buah Kelapa Tua Utuh / Parut Santan""","""Kelapa Setengah Tua Parut""","""Babay! Farm Jakarta"""
"""FRI-70030-00021-00003""","""Buah Segar Anggur Autumn USA FRESHGO""","""1kg""","""FreshGO Indonesia"""
"""FRI-70030-00241-00002""","""Anggur Shine Muscat import | FRESHGO""","""1/4 kg""","""FreshGO Indonesia"""
"""HAM-70080-00993-00010""","""HasilSumberalaM Anggur Import Manis Segar Shine Muscat Autumn Royal Red Globe Crimson Cand…","""Candy Snaps Permen 250gr""","""HasilSumberalaM"""
"""HAM-70080-00993-00011""","""HasilSumberalaM Anggur Import Manis Segar Shine Muscat Autumn Royal Red Globe Crimson Cand…","""Candy Snaps Permen 500gr""","""HasilSumberalaM"""
"""HAM-70080-00993-00012""","""HasilSumberalaM Anggur Import Manis Segar Shine Muscat Autumn Royal Red Globe Crimson Cand…","""Candy Snaps Permen 1kg""","""HasilSumberalaM"""
"""HAM-70126-00017-00005""","""HarvestMoon Nanas Honi Sunpride Per Pcs""","""Potong""","""Harvest Moon Jakarta"""
"""HAM-70126-00596-00002""","""HarvestMoon Kiwi Gold Zespri New Zealand per Kg""","""Jumbo""","""Harvest Moon Jakarta"""
"""IBF-60022-00179-00001""","""Buah Jeruk Impor Sunkist Lemon Santang Ponkan Wogan Murcot Mandarin :""","""Jeruk Afourer""","""IBAZA fruitsBOX"""


column,column_0,column_1
str,str,str
"""tanggal""","""2026-11-09""","""2026-11-09"""
"""produk_id""","""DOD-70072-00076""","""DOD-70072-00076"""
"""item_sku""","""DOD-70072-00076-00001""","""DOD-70072-00076-00002"""
"""opsi_default""","""true""","""false"""
"""nama""","""Buah Mangga Manalagi segar""","""Buah Mangga Manalagi segar"""
"""variant""","""1/2 kg""","""1 kg"""
"""harga""","""13000""","""25000"""
"""harga_coret""","""13000""","""25000"""
"""diskon_persen""","""0""","""0"""


In [13]:
terbaru.head()

tanggal,produk_id,item_sku,opsi_default,nama,variant,harga,harga_coret,diskon_persen,harga_voucher,harga_coret_voucher,diskon_voucher_persen,kampanye,harga_sebelum_kampanye,merchant_code,penjual,resmi,lokasi,lencana_toko,rating_toko,merek,kategori,kategori_tarik,rating,jumlah_ulasan,total_terjual,terjual_bulan_ini_min,url,sku,berat_gram,sumber_berat,jenis_satuan,jumlah_buah,berat_dikali
date,str,str,bool,str,str,i64,i64,i64,i64,i64,i64,str,i64,str,str,bool,str,str,f64,str,str,str,f64,i64,i64,i64,str,str,i64,str,str,i64,bool
2026-11-09,"""HYP-60026-23863""","""HYP-60026-23863-00001""",true,"""MANGGA INDRAMAYU 720 - 800 GR""",null,15920,19920,20,null,null,null,null,15920,"""HYP-60026""","""Hypermart Flagship Store""",true,"""Kota Surabaya""","""Diamond""",4.8,"""Hypermart""","""Bliblimart > Sayuran & Buah - buahan > Buah - buahan > Buah Lokal""","""BU-1000190""",4.6,10,79,50,"""https://www.blibli.com/p/mangga-indramayu-720-800-gr/ps--HYP-60026-23863?ds=HYP-60026-2386…","""HYP-60026-23863""",760,"""nama""","""berat""",null,false
2026-11-09,"""HYP-60026-20842""","""HYP-60026-20842-00001""",true,"""JERUK SANTANG DAUN 500-520 GR""",null,31670,31670,0,null,null,null,null,31670,"""HYP-60026""","""Hypermart Flagship Store""",true,"""Kota Surabaya""","""Diamond""",4.8,"""Hypermart""","""Bliblimart > Sayuran & Buah - buahan > Buah - buahan > Buah Lokal""","""BU-1000190""",4.5,87,925,50,"""https://www.blibli.com/p/jeruk-santang-daun-500-520-gr/ps--HYP-60026-20842?ds=HYP-60026-20…","""HYP-60026-20842""",510,"""nama""","""berat""",null,false
2026-11-09,"""HYP-60026-00639""","""HYP-60026-00639-00001""",true,"""NANAS GOLDEN HONI SUNPRIDE""",null,26900,35900,25,null,null,null,null,26900,"""HYP-60026""","""Hypermart Flagship Store""",true,"""Kota Surabaya""","""Diamond""",4.8,"""Hypermart""","""Bliblimart > Sayuran & Buah - buahan > Buah - buahan > Buah Lokal""","""BU-1000190""",4.7,158,1734,null,"""https://www.blibli.com/p/nanas-golden-honi-sunpride/ps--HYP-60026-00639?ds=HYP-60026-00639…","""HYP-60026-00639""",null,null,"""lain""",null,false
2026-11-09,"""HYP-60026-23792""","""HYP-60026-23792-00001""",true,"""APEL ENVY 540 - 600 GR""",null,55160,55160,0,null,null,null,null,55160,"""HYP-60026""","""Hypermart Flagship Store""",true,"""Kota Surabaya""","""Diamond""",4.8,"""Hypermart""","""Bliblimart > Sayuran & Buah - buahan > Buah - buahan > Buah Lokal""","""BU-1000190""",2.6,3,21,null,"""https://www.blibli.com/p/apel-envy-540-600-gr/ps--HYP-60026-23792?ds=HYP-60026-23792-00001""","""HYP-60026-23792""",570,"""nama""","""berat""",null,false
2026-11-09,"""RAM-70107-09255""","""RAM-70107-09255-00001""",true,"""MELON AMIKWA 1500 - 1750 G""",null,83200,83200,0,null,null,null,null,83200,"""RAM-70107""","""Ranch Market Flagship Store""",true,"""Kota Surabaya""","""Gold""",4.8,"""Ranch Market""","""Bliblimart > Sayuran & Buah - buahan > Buah - buahan > Buah Lokal""","""BU-1000190""",5.0,2,45,null,"""https://www.blibli.com/p/melon-amikwa-1500-1750-g/ps--RAM-70107-09255?ds=RAM-70107-09255-0…","""RAM-70107-09255""",1625,"""nama""","""berat""",null,false


## Task 5 (sebelum kode): tinjau harga coret Tokopedia (pertanyaan 1)

Tiap kartu hasil cari Tokopedia punya `price = {number, original, discountPercentage}`:
- `harga` ← `number` (harga bayar)
- `harga_asli` ← `original` (harga coret, sering kosong)
- `diskon_tokopedia` ← `discountPercentage` (selalu 0?)
- `diskon_hitung` = (harga_asli − harga) / harga_asli × 100

Tabel di bawah **bukan** hasil transform; hanya kolom harga dari data mentah, satu baris per (tanggal, produk).

In [14]:
from datetime import date
baris = []
for folder in sorted(Path("data_lake/raw/tokopedia").glob("date=*")):
    tanggal = date.fromisoformat(folder.name.removeprefix("date="))
    for berkas in sorted(folder.glob("page_*.json")):
        isi = json.load(open(berkas, encoding="utf-8"))
        if isinstance(isi, list):
            isi = isi[0]
        for p in isi["data"]["searchProductV5"]["data"]["products"]:
            harga = p.get("price") or {}
            baris.append({
                "tanggal": tanggal,
                "produk_id": str(p["id"]),
                "nama": p["name"],
                "toko": (p.get("shop") or {}).get("name"),
                "harga": harga.get("number"),
                "harga_asli": int("".join(c for c in harga.get("original") or "" if c.isdigit()) or 0) or None,
                "diskon_tokopedia": harga.get("discountPercentage"),
                "url": p["url"].split("?")[0],
            })
toko_harga = (
    pl.DataFrame(baris)
    .unique(["tanggal", "produk_id"], keep="first", maintain_order=True)
    .with_columns(
        selisih=pl.col("harga_asli") - pl.col("harga"),
        diskon_hitung=((pl.col("harga_asli") - pl.col("harga")) * 100 / pl.col("harga_asli")).round(1),
        manalagi=pl.col("nama").map_elements(parser.mangga_manalagi, return_dtype=pl.Boolean),
    )
)
print(toko_harga.height, "baris")
toko_harga.group_by("tanggal").agg(
    baris=pl.len(),
    ada_harga_asli=pl.col("harga_asli").is_not_null().sum(),
    diskon_tokopedia_bukan_0=(pl.col("diskon_tokopedia") != 0).sum(),
).sort("tanggal")

83317 baris


tanggal,baris,ada_harga_asli,diskon_tokopedia_bukan_0
date,u32,u32,u32
2026-10-23,5745,1784,0
2026-10-27,5669,1173,0
2026-10-28,5666,1174,0
2026-10-29,5625,1098,0
2026-10-30,4725,1041,0
2026-10-31,5635,1087,0
2026-11-01,5599,1080,0
2026-11-02,5532,1061,0
2026-11-03,5676,1094,0


### Per produk: dicoret selalu, kadang, atau tidak pernah?

- **selalu dicoret**: harga coret tiap hari → curiga diskon palsu (cek halaman produknya).
- **kadang dicoret**: harga coret hanya beberapa hari → kemungkinan diskon sungguhan.

Ganti `hanya_manalagi = False` untuk melihat semua produk.

In [15]:
hanya_manalagi = True   # False = semua produk Tokopedia

per_produk = (
    toko_harga.filter(pl.col("manalagi") | (not hanya_manalagi))
    .group_by("produk_id")
    .agg(
        nama=pl.col("nama").first(),
        toko=pl.col("toko").first(),
        hari_muncul=pl.len(),
        hari_dicoret=pl.col("harga_asli").is_not_null().sum(),
        harga_min=pl.col("harga").min(),
        harga_max=pl.col("harga").max(),
        diskon_hitung_max=pl.col("diskon_hitung").max(),
        url=pl.col("url").first(),
    )
    .with_columns(
        pola=pl.when(pl.col("hari_dicoret") == 0).then(pl.lit("tidak pernah dicoret"))
        .when(pl.col("hari_dicoret") == pl.col("hari_muncul")).then(pl.lit("selalu dicoret"))
        .otherwise(pl.lit("kadang dicoret"))
    )
    .sort("pola", "hari_dicoret", descending=[False, True])
)
display(per_produk.group_by("pola").agg(produk=pl.len()).sort("pola"))
per_produk

pola,produk
str,u32
"""selalu dicoret""",2
"""tidak pernah dicoret""",39


produk_id,nama,toko,hari_muncul,hari_dicoret,harga_min,harga_max,diskon_hitung_max,url,pola
str,str,str,u32,u32,i64,i64,f64,str,str
"""1256380209""","""Mangga Manalagi 500 Gr Fresh Murah""","""Aneka Sayuran Segar""",15,15,24750,24750,1.0,"""https://www.tokopedia.com/sayuransegarjkt/mangga-manalagi-500-gr-fresh-murah""","""selalu dicoret"""
"""8061245026""","""MANGGA MANALAGI MADU MANGGA MANALAGI MATANG MANIS Buah""","""Mantra Khatulistiwa""",15,15,14550,29100,3.0,"""https://www.tokopedia.com/mantra-khatulistiwa/mangga-manalagi-madu-mangga-manalagi-matang-…","""selalu dicoret"""
"""2064174592""","""Mangga Manalagi 900g - 1 Kg""","""ayoofresh""",1,0,19000,19000,null,"""https://www.tokopedia.com/ayoo-fresh/mangga-manalagi-950-g-1-kg-1731468922852246731""","""tidak pernah dicoret"""
"""9640686515""","""mangga manalagi madu 1 kg buah mangga manalagi""","""allfreshbuah02""",7,0,38000,38000,null,"""https://www.tokopedia.com/allfreshbuah02/mangga-manalagi-madu-1-kg-buah-mangga-manalagi""","""tidak pernah dicoret"""
"""104100829444""","""Mangga Madu Manalagi Fresh - Buah Segar dengan Rasa Manis dan Nikmat""","""Kebun Muria""",8,0,16500,16500,null,"""https://www.tokopedia.com/kebun-muria/mangga-madu-manalagi-fresh-buah-segar-dengan-rasa-ma…","""tidak pernah dicoret"""
"""10052736021""","""Mangga Manalagi Madu""","""Siti Fresh""",8,0,25000,25000,null,"""https://www.tokopedia.com/sitifresh/mangga-manalagi-madu-1-kg-aafd8""","""tidak pernah dicoret"""
"""11808381073""","""MANALAGI MANGGA | 1KG | NUSANTARA MARKET | SURABAYA""","""NUSANTARA.MARKET""",15,0,35000,35000,null,"""https://www.tokopedia.com/nusantaramarket/manalagi-mangga-masak-pohon-1kg-nusantara-market…","""tidak pernah dicoret"""
"""2562174962""","""buah mangga simanalagi / manalagi 1kg buah segar""","""Masak Sayur Bandung""",5,0,28900,28900,null,"""https://www.tokopedia.com/masaksayur/buah-mangga-simanalagi-manalagi-1kg-buah-segar""","""tidak pernah dicoret"""
"""102553764356""","""mangga manalagi jumbo isi 2 sampai 3 mentah tua rasa manis mantap""","""ASIZEHSHOP 4""",1,0,25999,25999,null,"""https://www.tokopedia.com/asizehshop-4/mangga-manalagi-jumbo-isi-3-mentah-tua-rasa-manis-m…","""tidak pernah dicoret"""


### Satu produk dari hari ke hari

Ambil `produk_id` dari tabel di atas, lalu bandingkan dengan halaman produknya (`url`).

In [16]:
contoh_id = "7286804348"   # ganti dengan produk_id dari tabel di atas
toko_harga.filter(pl.col("produk_id") == contoh_id).select(
    "tanggal", "harga", "harga_asli", "selisih", "diskon_hitung", "diskon_tokopedia"
).sort("tanggal")

tanggal,harga,harga_asli,selisih,diskon_hitung,diskon_tokopedia
date,i64,i64,i64,f64,i64
2026-10-23,250800,285000,34200,12.0,0
2026-10-27,285000,null,null,null,0
2026-10-28,285000,null,null,null,0
2026-10-29,285000,null,null,null,0
2026-10-30,285000,null,null,null,0
2026-10-31,285000,null,null,null,0
2026-11-01,285000,null,null,null,0
2026-11-02,285000,null,null,null,0
2026-11-04,285000,null,null,null,0


## Task 5: transform Tokopedia (`transform/tokopedia.py`)

Semua berkas mentah Tokopedia satu tanggal → tabel `tokopedia_produk`, **satu baris per opsi** (sama seperti Blibli, spec 5.0):
- baris kartu (`opsi_default = True`) dari hasil cari;
- baris opsi lain dari halaman produk `halaman/<id>.json` (hanya Mangga Manalagi, mulai 8 Nov);
- berat: **opsi → halaman (hanya kalau nama tanpa berat) → nama → URL**;
- `harga_asli` disimpan tapi tidak dipakai di fakta, `diskon_persen` selalu NULL; hanya salinan laptop.

### Kode

In [17]:
from transform import tokopedia
importlib.reload(tokopedia)
tampilkan_kode(tokopedia)

### Hasil: semua tanggal

In [18]:
from datetime import date

tokopedia_produk = pl.concat([
    tokopedia.ubah(folder, date.fromisoformat(folder.name.removeprefix("date=")))
    for folder in sorted(Path("data_lake/raw/tokopedia").glob("date=*"))
])
print(tokopedia_produk.height, "baris,", tokopedia_produk.width, "kolom")
tokopedia_produk.group_by("tanggal").agg(
    baris=pl.len(),
    opsi_lain=(~pl.col("opsi_default")).sum(),
    berat_terbaca=pl.col("berat_gram").is_not_null().sum(),
    kota_null=pl.col("kota").is_null().sum(),
).sort("tanggal")

83334 baris, 22 kolom


tanggal,baris,opsi_lain,berat_terbaca,kota_null
date,u32,u32,u32,u32
2026-10-23,5745,0,3776,7
2026-10-27,5669,0,3741,4
2026-10-28,5666,0,3755,2
2026-10-29,5625,0,3708,1
2026-10-30,4725,0,3175,3
2026-10-31,5635,0,3729,2
2026-11-01,5599,0,3722,0
2026-11-02,5532,0,3777,1
2026-11-03,5676,0,3847,1


### Yang perlu ditinjau

1. **Mangga Manalagi, tanggal terbaru**, urut harga per kg: berat dan `sumber_berat` masuk akal? Cocokkan beberapa dengan halaman produknya (`url`).
2. **Satu produk bervarian lengkap**: baris kartu + semua opsinya.

In [19]:
terbaru_tp = tokopedia_produk.filter(pl.col("tanggal") == pl.col("tanggal").max())
manalagi_tp = (
    terbaru_tp.filter(pl.col("nama").map_elements(parser.mangga_manalagi, return_dtype=pl.Boolean))
    .with_columns(harga_per_kg=(pl.col("harga") * 1000 / pl.col("berat_gram")).round())
)
display(manalagi_tp.group_by("sumber_berat").agg(baris=pl.len()))
display(manalagi_tp.select("produk_id", "opsi_default", "nama", "variant", "harga", "berat_gram",
                           "sumber_berat", "harga_per_kg", "nama_toko", "url").sort("harga_per_kg"))
terbaru_tp.filter(pl.col("induk_id") == "103388173219").select(
    "produk_id", "opsi_default", "variant", "harga", "berat_gram", "sumber_berat", "url")

sumber_berat,baris
str,u32
"""halaman""",2
"""nama""",10
"""opsi""",10
"""url""",1


produk_id,opsi_default,nama,variant,harga,berat_gram,sumber_berat,harga_per_kg,nama_toko,url
str,bool,str,str,i64,i64,str,f64,str,str
"""102383650601""",true,"""Mangga Manalagi Segar 1Kg – Manis dan Daging Tebal""",null,14000,1000,"""nama""",14000.0,"""Mracang Market""","""https://www.tokopedia.com/mracangmarket/mangga-manalagi-segar-1kg-manis-dan-daging-tebal-1…"
"""104001505646""",false,"""1 Kg Mangga MANALAGI MADU Manis Legit dikirim Mengkal""","""1Kg Manalagi isi 4-5""",15999,1000,"""opsi""",15999.0,"""MURAHMURAH.KEDIRI3""","""https://www.tokopedia.com/murahmurahkediri3/1-kg-mangga-manalagi-madu-manis-legit-dikirim-…"
"""104001505645""",true,"""1 Kg Mangga MANALAGI MADU Manis Legit dikirim Mengkal""","""1Kg Manalagi isi 3""",19999,1000,"""opsi""",19999.0,"""MURAHMURAH.KEDIRI3""","""https://www.tokopedia.com/murahmurahkediri3/1-kg-mangga-manalagi-madu-manis-legit-dikirim-…"
"""104132111950""",false,"""MANGGA MANALAGI MADU (1 KG) - MANIS LEGIT""","""5 Kg""",114500,5000,"""opsi""",22900.0,"""tukatukushop.id""","""https://www.tokopedia.com/tukatukushop/mangga-manalagi-madu-1-kg-manis-legit-1738100287567…"
"""104132111949""",false,"""MANGGA MANALAGI MADU (1 KG) - MANIS LEGIT""","""3 Kg""",69500,3000,"""opsi""",23167.0,"""tukatukushop.id""","""https://www.tokopedia.com/tukatukushop/mangga-manalagi-madu-1-kg-manis-legit-1738100287567…"
"""104132111948""",true,"""MANGGA MANALAGI MADU (1 KG) - MANIS LEGIT""","""1 Kg""",24500,1000,"""opsi""",24500.0,"""tukatukushop.id""","""https://www.tokopedia.com/tukatukushop/mangga-manalagi-madu-1-kg-manis-legit-1738100287566…"
"""10052736021""",true,"""Mangga Manalagi Madu""",null,25000,1000,"""url""",25000.0,"""Siti Fresh""","""https://www.tokopedia.com/sitifresh/mangga-manalagi-madu-1-kg-aafd8"""
"""6690740844""",true,"""Mangga Manalagi Madu""","""1 kg""",25000,1000,"""opsi""",25000.0,"""Toko Buah Sumber Sejahtera""","""https://www.tokopedia.com/tokobuahsumbersejahtera/mangga-manalagi-madu-1-kg"""
"""104132127302""",false,"""MANGGA MANALAGI MADU (1 KG) - MANIS LEGIT""","""1/2 Kg""",13500,500,"""opsi""",27000.0,"""tukatukushop.id""","""https://www.tokopedia.com/tukatukushop/mangga-manalagi-madu-1-kg-manis-legit-1738100317727…"


produk_id,opsi_default,variant,harga,berat_gram,sumber_berat,url
str,bool,str,i64,i64,str,str


## Task 6: validasi + karantina (`transform/skema.py`)

Aturan: spec bagian 6. `pisahkan(df, Skema, sumber)` → `(lolos, karantina)`:
- **Gagal cek** (harga ≤ 0 atau NULL, rating di luar 0–5, diskon di luar 0–100, baris ganda, cek A–G Blibli) → masuk karantina, **tidak** lolos.
- **Cek H** (`berat_dikali = true`, multipack "8x125GR") → **disalin** ke karantina, **tetap lolos**.
- **Kolom hilang / tipe salah** → error dilempar (kode atau bentuk data berubah), tidak dikarantina.
- Cek A–G dilewati kalau kolom yang dibandingkan NULL (opsi lain Blibli tidak punya voucher/kampanye/sku).

### Kode


In [20]:
from transform import skema
importlib.reload(skema)
tampilkan_kode(skema)


### Hasil: semua tanggal


In [21]:
from datetime import date
from transform import blibli, tokopedia


def semua_tanggal(modul, sumber):
    return pl.concat([
        modul.ubah(folder, date.fromisoformat(folder.name.removeprefix("date=")))
        for folder in sorted(Path(f"data_lake/raw/{sumber}").glob("date=*"))
    ])


hasil = {}
for sumber, modul, skema_ in (("blibli", blibli, skema.SkemaBlibli),
                              ("tokopedia", tokopedia, skema.SkemaTokopedia)):
    df = semua_tanggal(modul, sumber)
    hasil[sumber] = skema.pisahkan(df, skema_, sumber)
    print(f"{sumber}: {df.height} baris, lolos {hasil[sumber][0].height}, karantina {hasil[sumber][1].height}")

karantina = pl.concat([k for _, k in hasil.values()])
karantina.with_columns(pl.col("alasan").str.split("; ")).explode("alasan").group_by("sumber", "alasan").agg(
    baris=pl.len(), produk=pl.col("produk_id").n_unique(), tanggal=pl.col("tanggal").n_unique()
).sort("sumber", "baris", descending=[False, True])


blibli: 26535 baris, lolos 26401, karantina 181
tokopedia: 83334 baris, lolos 83334, karantina 44


sumber,alasan,baris,produk,tanggal
str,str,u32,u32,u32
"""blibli""","""harga: not_nullable""",126,28,3
"""blibli""","""cek_H_berat_dikali""",48,6,16
"""blibli""","""cek_F_kampanye""",8,2,7
"""tokopedia""","""cek_H_berat_dikali""",44,4,15


### Yang perlu ditinjau

1. **Cek F** (8 baris, 2 produk): harga turun dari "sebelum kampanye" tanpa nama kampanye. Spec 6.1 mencatat ini sebagai kasus asli (mangga Rp10.000, diskon 60–75%). Tetap dikarantina?
2. **Cek H** (multipack): apakah `berat_gram` sudah total (misalnya 8 × 125 = 1000)? Baris ini tetap masuk fakta.
3. **Harga NULL**: semuanya opsi Blibli yang tidak tersedia (keputusan B)?


In [22]:
def isi(alasan, *kolom):
    """Baris karantina dengan `alasan` tertentu, kolom asli dibaca dari `data`."""
    k = karantina.filter(pl.col("alasan").str.contains(alasan))
    return pl.DataFrame([{"sumber": s, **{c: json.loads(d).get(c) for c in kolom}}
                         for s, d in zip(k["sumber"], k["data"])])


print("1. cek F")
display(isi("cek_F", "tanggal", "produk_id", "nama", "harga", "harga_coret", "diskon_persen",
            "harga_sebelum_kampanye", "kampanye"))
print("2. cek H (nama unik)")
display(isi("cek_H", "nama", "variant", "berat_gram", "jumlah_buah").unique(maintain_order=True))
print("3. harga NULL")
display(isi("harga: not_nullable", "tanggal", "item_sku", "nama", "variant").group_by("tanggal").len().sort("tanggal"))


1. cek F


sumber,tanggal,produk_id,nama,harga,harga_coret,diskon_persen,harga_sebelum_kampanye,kampanye
str,str,str,str,i64,i64,i64,i64,null
"""blibli""","""2026-10-18""","""SES-70872-06867""","""Mangga Harum Manis ( 0.9 - 1.1 kg / pack)""",10000,25000,60,17500,null
"""blibli""","""2026-10-19""","""SES-70872-06867""","""Mangga Harum Manis ( 0.9 - 1.1 kg / pack)""",10000,25000,60,17500,null
"""blibli""","""2026-10-19""","""FRX-60021-01718""","""Mangga Harum Manis 1Kg Freshbox""",10000,40000,75,15000,null
"""blibli""","""2026-10-27""","""FRX-60021-01718""","""Mangga Harum Manis 1Kg Freshbox""",10000,40000,75,15000,null
"""blibli""","""2026-10-28""","""FRX-60021-01718""","""Mangga Harum Manis 1Kg Freshbox""",10000,40000,75,15000,null
"""blibli""","""2026-10-29""","""FRX-60021-01718""","""Mangga Harum Manis 1Kg Freshbox""",10000,40000,75,15000,null
"""blibli""","""2026-10-30""","""FRX-60021-01718""","""Mangga Harum Manis 1Kg Freshbox""",10000,40000,75,15000,null
"""blibli""","""2026-10-31""","""FRX-60021-01718""","""Mangga Harum Manis 1Kg Freshbox""",10000,40000,75,15000,null


2. cek H (nama unik)


sumber,nama,variant,berat_gram,jumlah_buah
str,str,str,i64,null
"""blibli""","""BLUEBERRY TRAIN TOY 8x125GR PACK""",null,1000,null
"""blibli""","""Sunsweet Buah Prunes 200 gr×12 Pcs""",null,2400,null
"""blibli""","""Kelengkeng Impor""","""Bundle 3 x 250 gr""",750,null
"""blibli""","""Apel Fuji Impor""","""Bundle 2 x 500 gr""",1000,null
"""blibli""","""Apel Fuji Impor""","""Bundle 3 x 500 gr""",1500,null
"""blibli""","""Jeruk Lemon Impor""","""Bundle 2 x 500 gram""",1000,null
"""blibli""","""Jeruk Lemon Impor""","""Bundle 3 x 500 gram""",1500,null
"""blibli""","""Jeruk Siam""","""Bundle 3 x 500 gr""",1500,null
"""blibli""","""Jeruk Siam""","""Bundle 2 x 500 gr""",1000,null


3. harga NULL


tanggal,len
str,u32
"""2026-11-07""",43
"""2026-11-08""",43
"""2026-11-09""",40


## Task 7: star schema (`transform/model.py`)

Aturan: spec 5.3. `bangun(blibli, tokopedia)` → `Bintang(dim_produk, dim_toko, dim_tanggal, fakta_harga)`:
- **Grain fakta**: satu opsi produk per marketplace per tanggal (1 baris lapisan 1 = 1 baris fakta).
- **Kunci**: `md5("<sumber>|<id>")`, dihitung langsung dari ID lapisan 1, jadi toko tanpa kota tetap masuk fakta.
- **Tokopedia, keputusan 9 Nov**: opsi satu produk = satu produk induk (`induk_id`), seperti Blibli. Halaman gagal di satu tanggal → induk dari tanggal lain.
- **`harga_per_kg`**: kalau `jenis_satuan = berat`; **`harga_per_buah`**: kalau `jumlah`. Selain itu NULL.
- **Batas karantina fakta, keputusan 9 Nov (13 #5)**: Rp2.000–2.000.000 per kg.
- Tokopedia `harga_coret` dan `diskon_persen` = NULL (13 #1–2). Dimensi: tanggal terakhir menang (SCD 1).

### Kode


In [23]:
from transform import model
importlib.reload(model)
tampilkan_kode(model)


### Hasil: semua tanggal (pakai hasil `lolos` dari sel Task 6, jalankan sel itu dulu)


In [24]:
bintang = model.bangun(hasil["blibli"][0], hasil["tokopedia"][0])
for nama in ("dim_produk", "dim_toko", "dim_tanggal", "fakta_harga"):
    print(f"{nama}: {getattr(bintang, nama).height} baris")

fakta_lolos, fakta_karantina = skema.pisahkan(bintang.fakta_harga, skema.SkemaFakta, "campuran")
print("fakta karantina:", fakta_karantina.height)
bintang.dim_tanggal


dim_produk: 10730 baris
dim_toko: 1599 baris
dim_tanggal: 17 baris
fakta_harga: 109735 baris
fakta karantina: 2665


tanggal,tahun,bulan,minggu,hari
date,i32,i8,i8,str
2026-10-18,2026,10,42,"""Minggu"""
2026-10-19,2026,10,43,"""Senin"""
2026-10-23,2026,10,43,"""Jumat"""
2026-10-27,2026,10,44,"""Selasa"""
2026-10-28,2026,10,44,"""Rabu"""
2026-10-29,2026,10,44,"""Kamis"""
2026-10-30,2026,10,44,"""Jumat"""
2026-10-31,2026,10,44,"""Sabtu"""
2026-11-01,2026,11,44,"""Minggu"""


### Yang perlu ditinjau

1. **Karantina fakta** (harga per kg di luar Rp2.000–2.000.000): harganya memang mahal, atau beratnya salah baca?
2. **Mangga Manalagi, tanggal terbaru**, per opsi: `harga_per_kg` masuk akal? Opsi Tokopedia berkumpul di satu produk induk?
3. **Asal Tokopedia**: sebagian besar `tidak_diketahui` (aturan nama hanya menangkap kata seperti "impor", "fuji", "manalagi").


In [25]:
DATA_FAKTA = {"produk_key": pl.Utf8, "harga": pl.Int64, "berat_gram": pl.Int64, "harga_per_kg": pl.Int64}
v = bintang.fakta_harga.join(bintang.dim_produk, on="produk_key").join(
    bintang.dim_toko.drop("sumber"), on="toko_key")

print("1. karantina fakta (per nama)")
display(fakta_karantina.with_columns(pl.col("data").str.json_decode(pl.Struct(DATA_FAKTA)).struct.field(*DATA_FAKTA))
        .join(bintang.dim_produk.select("produk_key", "nama"), on="produk_key")
        .group_by("nama").agg(pl.len(), pl.col("harga").median(), pl.col("berat_gram").first(),
                              pl.col("harga_per_kg").median()).sort("harga_per_kg", descending=True))

print("2. Mangga Manalagi, tanggal terbaru")
display(v.filter((pl.col("tanggal") == pl.col("tanggal").max())
                 & pl.col("nama").map_elements(parser.mangga_manalagi, return_dtype=pl.Boolean))
        .select("sumber", "produk_id", "item_sku", "opsi_default", "nama", "variant", "harga",
                "berat_gram", "harga_per_kg", "nama_toko", "region")
        .sort("sumber", "produk_id", "harga_per_kg"))

print("3. asal per sumber")
bintang.dim_produk.group_by("sumber", "asal").len().sort("sumber", "asal")


1. karantina fakta (per nama)


nama,len,harga,berat_gram,harga_per_kg
str,u32,f64,i64,f64
"""SU Brand Manisan Buah Kundur 250g - Medium Pack - 50Pcs""",16,1.253105e6,250,5.01242e6
"""persimon korea original kesemek import korea - 1kg""",13,4.32e6,1000,4.32e6
"""buah anggur nagano asli jepang / anggur import jepang original 700gr""",15,2.815e6,700,4.021429e6
"""Buah Strawberry Pink Snow Fresh 350Gr import Korea ORIGINAL""",14,1.3909e6,350,3.974e6
"""Kurma Ajwa Aliyah Organik 500gr - 1dus - Kurma Ajwa Aliyah Castle""",14,1.8e6,500,3.6e6
"""Pear RRC Xiang Lie 120gr Per Carton""",15,280000.0,120,2.333333e6
"""Dried lime 100gr - Slice Jeruk Nipis kering iris - Sona""",12,220000.0,100,2.2e6
"""Yellow Peach Shuimitao Buah Persik Kuning RRC Box Isi 4 Berat Total 600gr""",14,1.262e6,600,2.103333e6
"""Kurma Date Crown Lulu 250 g - 1 Dus (isi 20 pcs)""",26,504000.0,250,2.016e6


2. Mangga Manalagi, tanggal terbaru


sumber,produk_id,item_sku,opsi_default,nama,variant,harga,berat_gram,harga_per_kg,nama_toko,region
str,str,str,bool,str,str,i64,i64,i64,str,str
"""blibli""","""DOD-70072-00076""","""DOD-70072-00076-00002""",false,"""Buah Mangga Manalagi segar""","""1 kg""",25000,1000,25000,"""Doyan Daging Store Depok""","""jabodetabek"""
"""blibli""","""DOD-70072-00076""","""DOD-70072-00076-00001""",true,"""Buah Mangga Manalagi segar""","""1/2 kg""",13000,500,26000,"""Doyan Daging Store Depok""","""jabodetabek"""
"""blibli""","""RAM-70107-29314""","""RAM-70107-29314-00001""",true,"""MANGGA MANALAGI 170 - 180 G""",null,6800,175,38857,"""Ranch Market Flagship Store""","""jawa_lain"""
"""blibli""","""SAX-70009-02879""","""SAX-70009-02879-00001""",true,"""Mangga Manalagi""","""1 kg""",25700,1000,25700,"""Sayurbox Flagship Store""","""jabodetabek"""
"""tokopedia""","""10052735997""","""10052736021""",true,"""Mangga Manalagi Madu""",null,25000,1000,25000,"""Siti Fresh""","""jabodetabek"""
"""tokopedia""","""102383650601""","""102383650601""",true,"""Mangga Manalagi Segar 1Kg – Manis dan Daging Tebal""",null,14000,1000,14000,"""Mracang Market""","""jawa_lain"""
"""tokopedia""","""102472942809""","""102472942809""",true,"""MANGGA MANALAGI MANIS, (1KG) flash sale Buah""",null,45500,1000,45500,"""MAS JOKO BUAH SEGAR""","""jabodetabek"""
"""tokopedia""","""1025421585""","""104132111950""",false,"""MANGGA MANALAGI MADU (1 KG) - MANIS LEGIT""","""5 Kg""",114500,5000,22900,"""tukatukushop.id""","""jabodetabek"""
"""tokopedia""","""1025421585""","""104132111949""",false,"""MANGGA MANALAGI MADU (1 KG) - MANIS LEGIT""","""3 Kg""",69500,3000,23167,"""tukatukushop.id""","""jabodetabek"""


3. asal per sumber


sumber,asal,len
str,str,u32
"""blibli""","""impor""",1079
"""blibli""","""lokal""",1825
"""tokopedia""","""impor""",1179
"""tokopedia""","""lokal""",667
"""tokopedia""","""tidak_diketahui""",5980
